In [3]:
import sys
from pathlib import Path

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

this_path = Path(__file__) if '__file__' in globals() else Path("<unknown>.ipynb").resolve()
work_path = next((p for p in this_path.parents if p.name == "research"), None)
tools_path = str(work_path / Path("../torch-tools"))
sys.path.append(tools_path)

from run_manager import RunViewer


In [4]:
nb_path = Path().resolve()
rv = RunViewer(exp_path=nb_path)
df_base = rv.fetch_results(met_listed=False, refresh=True)


In [5]:
df = df_base
df = df.with_columns((~pl.col("model_arc").str.contains("def")).alias("weight_unified"))
df = df.with_columns((pl.col("model_arc").str.contains("bn")).alias("bn_stat_disabled"))
# df = df.filter(pl.col("optimizer").is_null())

col_name = "val_acc"
# col_name = "duration"
df = df.group_by(["model_arc", "ensemble_type"], maintain_order=True).agg(
        pl.col(col_name).mean().alias("acc_mean"), 
        pl.col(col_name).min().alias("acc_min"), 
        pl.col(col_name).max().alias("acc_max"), 
        pl.col(col_name).var().alias("acc_var"),
        pl.col("duration").mean().alias("duration"),
        pl.col("grad_mean").mean().alias("grad_mean_mean"),
        pl.col("grad_mean").min().alias("grad_mean_min"),
        pl.col("grad_mean").max().alias("grad_mean_max"),
        pl.col("weight_unified").first().alias("weight_unified"),
        pl.col("bn_stat_disabled").first().alias("bn_stat_disabled"),
        pl.col(col_name).count().alias("trials")
    )
display(df)

# df = df.pivot(values="acc_mean", index=["model_arc"], on="ensemble_type", sort_columns=True, aggregate_function="mean")
df = df.pivot(values="acc_mean", index=["model_arc", "weight_unified", "bn_stat_disabled"], on="ensemble_type", sort_columns=True, aggregate_function="mean")
df = df.with_columns((pl.col("EE") - pl.col("ME")).alias("EE - ME"))
# df = df.with_columns((pl.col("EE") / pl.col("ME")).alias("EE / ME"))
# df = df.with_columns((((pl.col("EE") - pl.col("ME")) * 100).round(2).cast(pl.Utf8) + "%").alias("diff"))
display(df)

model_arc,ensemble_type,acc_mean,acc_min,acc_max,acc_var,duration,grad_mean_mean,grad_mean_min,grad_mean_max,weight_unified,bn_stat_disabled,trials
str,str,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,u32
"""models.resnet_git_ee_def resne…","""EE""",0.689875,0.6774,0.6994,0.00006,343.633145,0.000282,0.000104,0.000481,false,false,8
"""models.resnet_git_ee_def resne…","""ME""",0.694875,0.6856,0.7096,0.000064,1439.00793,0.000215,0.000055,0.000493,false,false,8
"""models.resnet_git_ee resnet18""","""EE""",0.706975,0.7004,0.714,0.000022,343.748641,0.000314,0.000061,0.000431,true,false,8
"""models.resnet_git_ee resnet18""","""ME""",0.697914,0.6942,0.707,0.000025,1443.475283,0.000224,0.000083,0.000355,true,false,7
"""models.resnet_git_ee_bn resnet…","""EE""",0.677057,0.6594,0.6852,0.000074,344.928241,0.000375,0.000245,0.000473,true,true,7
"""models.resnet_git_ee_bn resnet…","""ME""",0.678714,0.6758,0.6858,0.000014,1399.051098,0.000182,0.000041,0.000378,true,true,7
"""models.resnet_git_ee_bn_def re…","""EE""",0.663114,0.6582,0.67,0.000017,344.437878,0.000296,0.000208,0.000442,false,true,7
"""models.resnet_git_ee_bn_def re…","""ME""",0.680857,0.672,0.695,0.000066,1402.606567,0.000231,0.000082,0.000469,false,true,7


model_arc,weight_unified,bn_stat_disabled,EE,ME,EE - ME
str,bool,bool,f64,f64,f64
"""models.resnet_git_ee_def resne…",false,false,0.689875,0.694875,-0.005
"""models.resnet_git_ee resnet18""",true,false,0.706975,0.697914,0.009061
"""models.resnet_git_ee_bn resnet…",true,true,0.677057,0.678714,-0.001657
"""models.resnet_git_ee_bn_def re…",false,true,0.663114,0.680857,-0.017743
